<a href="https://colab.research.google.com/github/MarkusThill/techdays26/blob/main/torch_ntuple_net.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

TODO:
- Build plots for BitBullyArena Results!

In [10]:
# Only relevant for Google Colab:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import output

    output.enable_custom_widget_manager()

In [11]:
if IN_COLAB:
    # Only relevant for Google Colab:
    !ssh-keygen -t rsa -b 4096 -f ~/.ssh/id_rsa -N "" -q
    !ssh-keyscan -t rsa github.com >> ~/.ssh/known_hosts
    !cat ~/.ssh/id_rsa.pub

Add key here:
https://github.com/settings/keys

In [12]:
if IN_COLAB:
    # Only relevant for Google Colab:
    !ssh -T git@github.com
    !git config --global user.email "8896197+MarkusThill@users.noreply.github.com"
    !git config --global user.name "Markus Thill"
    !git clone git@github.com:MarkusThill/techdays26.git
    !pip install -e techdays26[dev,lab1]

In [13]:
from techdays26.ntuples import NTUPLE_BITIDX_LIST

In [26]:
import torch
import torch.nn as nn


class TDConnect4AgentTorch:
    """TD n-tuple agent compatible with `Connect4Agent` Protocol.

    Implements:
      - score_all_moves(board) -> dict[int,int]
      - best_move(board) -> int
      - score_move(board, column, first_guess=0) -> int
    """

    def __init__(
        self,
        model_path: str | None = None,
        *,
        tie_break: str = "center",
    ) -> None:
        net2 = NTupleNetwork.load(model_path, device="cpu")
        net2.eval()
        self._tie_break = tie_break
        self._eval = net2

    # ---- Protocol method 1 ----
    def score_all_moves(self, board) -> dict[int, int]:
        """Return {col: score} for all legal moves. Illegal/full columns are excluded."""
        player_to_move = board.current_player() - 1  # BitBully: {1,2} -> {0,1}
        if player_to_move not in (0, 1):
            raise ValueError(f"Unexpected current_player(): {board.current_player()}")

        scores: dict[int, int] = {}

        for col in range(7):
            if not board.is_legal_move(col):
                continue

            score = self.score_move(board=board, column=col)
            scores[col] = score

        return scores

    # ---- Protocol method 2 ----
    def best_move(self, board) -> int:
        """Return best legal move using BitBully-like tie breaking."""
        scores = self.score_all_moves(board)
        if not scores:
            raise ValueError("No legal moves available.")

        best = max(scores.values())
        candidates = [c for c, v in scores.items() if v == best]

        if len(candidates) == 1:
            return candidates[0]

        if self._tie_break == "center":
            center = 3
            return min(candidates, key=lambda c: (abs(c - center), c))
        if self._tie_break == "left":
            return min(candidates)
        if self._tie_break == "right":
            return max(candidates)

        raise ValueError("tie_break must be one of: 'center', 'left', 'right'")

    # ---- Optional Protocol method ----
    def score_move(self, board, column: int, first_guess: int = 0) -> int:
        """Evaluate a single legal move. `first_guess` is accepted for Protocol compatibility."""
        _ = first_guess  # this TD agent ignores it
        if not board.is_legal_move(column):
            raise ValueError(f"Illegal move: column {column}")

        player_to_move = board.current_player()
        after = board.play_on_copy(column)

        if after.has_win():
            return 100

        all_tokens, active_tokens, moves_left = after._board.rawState()
        torch_board = BoardBatch(
            all_tokens=torch.tensor([all_tokens]),
            active_tokens=torch.tensor([active_tokens]),
            moves_left=torch.tensor([moves_left]),
        )

        # opponent can win
        if after.can_win_next():
            return -100
        
        score = float(self._eval.forward(torch_board)[0].item())

        if player_to_move == 2:
            score = -score

        return int(score * 100.0)


class NTupleNetwork(nn.Module):
    def __init__(self, n_tuple_list: list[list[int]]):
        super().__init__()

        self.M = len(n_tuple_list)
        self.N = len(n_tuple_list[0])
        self.K = 4**self.N

        # [M, N] bit indices
        self.n_tuple_tensor = torch.tensor(n_tuple_list, dtype=torch.int64)

        # Two players × M LUTs × K entries
        # 0 = Yellow, 1 = Red
        self.W = nn.Parameter(torch.zeros(2, self.M, self.K))

    def forward(self, board: "BoardBatch") -> torch.Tensor:
        """
        Returns:
            [B] tensor in [-1, 1]
        """
        # [B, M] table indices
        T = board.table_positions(self.n_tuple_tensor)
        T_mir = board.mirror().table_positions(self.n_tuple_tensor)
        B, M = T.shape
        dev = T.device

        # Active player per board: 0=Yellow, 1=Red
        # reuse your parity logic
        player_idx = ((board.moves_left.to(torch.int64) & 1) != 0).to(torch.int64)
        # shape: [B]

        # Pattern indices [M]
        m_idx = torch.arange(M, device=dev)

        # Gather:
        # W[player_idx[b], m, T[b,m]]
        w = self.W[
            player_idx.unsqueeze(1),  # [B,1]
            m_idx.unsqueeze(0),  # [1,M]
            T,  # [B,M]
        ]  # -> [B,M]
        w_mir = self.W[player_idx.unsqueeze(1), m_idx.unsqueeze(0), T_mir]

        # Sum over patterns (and symmetry): [B]
        score = (w + w_mir).sum(dim=1)
        return torch.tanh(score)

    @torch.no_grad()
    def save(self, path: str) -> None:
        """Save model weights + architecture-relevant metadata."""
        payload = {
            "state_dict": self.state_dict(),
            "n_tuple_list": self.n_tuple_tensor.cpu().tolist(),
        }
        torch.save(payload, path)

    @classmethod
    def load(
        cls,
        path: str,
        *,
        device: torch.device | str = "cpu",
        strict: bool = True,
    ) -> "NTupleNetwork":
        """
        Load model fully onto the specified device (CPU or GPU).
        """

        # 1) Always load checkpoint onto CPU first (safe & portable)
        payload = torch.load(path, map_location="cpu")

        if not isinstance(payload, dict) or "state_dict" not in payload:
            raise ValueError("Invalid checkpoint format.")

        n_tuple_list = payload.get("n_tuple_list")
        if n_tuple_list is None:
            raise ValueError("Checkpoint missing 'n_tuple_list'.")

        # 2) Construct model (still on CPU)
        model = cls(n_tuple_list=n_tuple_list)

        # 3) Load weights (still CPU tensors)
        model.load_state_dict(payload["state_dict"], strict=strict)

        # 4) Move EVERYTHING (parameters + buffers) in one go
        model = model.to(device)

        return model


import torch
import torch.nn.functional as F


def best_afterstate_values(
    board: "BoardBatch",
    net: "NTupleNetwork",
    *,
    moves_mask: torch.Tensor | None = None,
    randomize: torch.Tensor | None = None,  # [B] bool (epsilon-greedy)
    use_non_losing: bool = True,
    no_grad: bool = True,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Returns:
        chosen_mv:  [B] int64 one-hot move mask (landing square)
        chosen_val: [B] float32 value (reward if terminal else net score)
    """
    dev = board.all_tokens.device
    B = board.all_tokens.shape[0]

    # Which move set to iterate?
    if moves_mask is None:
        moves_mask = (
            board.generate_non_losing_moves()
            if use_non_losing
            else board.generate_legal_moves()
        )
    moves_mask = moves_mask.to(device=dev, dtype=torch.int64)

    yellow_to_move = (board.moves_left.to(torch.int64) & 1) == 0  # [B] bool

    neg_inf = torch.tensor(float("-inf"), device=dev)
    pos_inf = torch.tensor(float("inf"), device=dev)
    best_val = (
        torch
        .where(yellow_to_move, neg_inf, pos_inf)
        .to(torch.float32)
        .expand(B)
        .clone()
    )
    best_mv = torch.zeros((B,), device=dev, dtype=torch.int64)

    # Uniform random move via reservoir sampling over the iterated moves
    rand_mv = torch.zeros((B,), device=dev, dtype=torch.int64)
    rand_val = torch.full((B,), float("nan"), device=dev, dtype=torch.float32)
    seen = torch.zeros(
        (B,), device=dev, dtype=torch.int32
    )  # number of moves seen so far per board

    ctx = torch.no_grad() if no_grad else torch.enable_grad()
    with ctx:
        for mv in board.iter_move_masks(moves_mask):
            active = mv != 0
            if not active.any():
                break

            # Afterstate
            tmp = type(board)(
                all_tokens=board.all_tokens.clone(),
                active_tokens=board.active_tokens.clone(),
                moves_left=board.moves_left.clone(),
            )
            legal = tmp.play_masks(mv)
            active = active & legal  # defensive

            # Terminal reward: +1/-1/0 or NaN if not done
            r = tmp.reward().to(torch.float32)
            v = net(tmp).to(torch.float32)

            # tiebreak noise (optional)
            eps = 1e-4
            v = v + eps * torch.randn_like(v)

            val = torch.where(torch.isnan(r), v, r)  # [B]

            # --- greedy best update ---
            better = (
                torch.where(yellow_to_move, val > best_val, val < best_val) & active
            )
            best_val = torch.where(better, val, best_val)
            best_mv = torch.where(better, mv, best_mv)

            # --- reservoir sampling (uniform random among legal moves) ---
            # increment seen count for boards where this move exists
            seen = seen + active.to(seen.dtype)  # t in {1..}
            # replace current random choice with prob 1/seen
            # (uniform float u in [0,1); replace if u < 1/t)
            t = seen.to(torch.float32)
            replace = active & (torch.rand((B,), device=dev) < (1.0 / t))
            rand_mv = torch.where(replace, mv, rand_mv)
            rand_val = torch.where(replace, val, rand_val)

    if randomize is None:
        return best_mv, best_val

    randomize = randomize.to(device=dev, dtype=torch.bool)
    chosen_mv = torch.where(randomize, rand_mv, best_mv)
    chosen_val = torch.where(randomize, rand_val, best_val)
    return chosen_mv, chosen_val

In [15]:
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")

In [16]:
from bitbully import BitBully
from bitbully.agent_interface import Connect4Agent

bitbully_agent_max_depth = BitBully(opening_book="12-ply-dist", tie_break="random")

# Some weaker agents with limited depth
bitbully_agents = {}
for search_depth in (1, 2, 4, 8):
    agent = BitBully(opening_book=None, tie_break="random", max_depth=search_depth)
    bitbully_agents[f"bitbully-{search_depth}ply"] = agent
    agent = BitBully(opening_book="8-ply", tie_break="random", max_depth=search_depth)
    bitbully_agents[f"bitbully-{search_depth}-ply-book8ply"] = agent

bitbully_agents[f"bitbully-16ply-book12ply"] = BitBully(
    opening_book="12-ply-dist", tie_break="random", max_depth=16
)
bitbully_agents[f"bitbully-full-strength"] = bitbully_agent_max_depth

In [17]:
import logging
from techdays26 import bitbully_arena as ba

from techdays26.bitbully_arena import (
    #BitBullyArena,
    #ArenaConfig,
    #AgentSpec,
    #RandomAgent,
    #Color,
    #TimeControl,
    get_table_legend,
    format_aggregate_table,
)


def evaluate(td_agent, rnd_agent, bitbully_agents):
    # Logger is optional; arena is silent by default unless you configure logging.
    logger = logging.getLogger("bitbully.arena")
    logger.setLevel(logging.WARNING)  # warnings for illegal/exception/timeout

    agent_specs = [
        ba.AgentSpec(
            agent_id=k,
            agent=v,
            colors=(ba.Color.YELLOW, ba.Color.RED),  # can play both
            epsilons=(0.00, 0.1, 0.2, 0.3) if "full-strength" in k else (0.00,),
        )
        for k, v in bitbully_agents.items()
    ]
    matchups = (
        [ba.Matchup(yellow_id=k, red_id="ntuple") for k in bitbully_agents.keys()]
        + [ba.Matchup(yellow_id="ntuple", red_id=k) for k in bitbully_agents.keys()]
        + [
            ba.Matchup(yellow_id="ntuple", red_id="random"),
            ba.Matchup(yellow_id="random", red_id="ntuple"),
        ]
    )

    cfg = ba.ArenaConfig(
        agents=(
            *agent_specs,
            ba.AgentSpec(
                agent_id="random",
                agent=rnd_agent,
                colors=(ba.Color.YELLOW, ba.Color.RED),  # can play both
                epsilons=(0.00,),
            ),
            ba.AgentSpec(
                agent_id="ntuple",
                agent=td_agent,
                colors=(ba.Color.YELLOW, ba.Color.RED),  # can play both,
                epsilons=(0.00,),
            ),
        ),
        n_games=50,  # n games per pairing per ε per color-swap
        time_control=ba.TimeControl(
            per_move_timeout_s=4.0,  # best-effort (measured after return)
            per_game_budget_s=45.0,  # seconds of total think time
        ),
        matchups=matchups,
        seed=None,
        log_scores=False,  # set True to also call score_all_moves() for logging
        use_tqdm=True,  # optional progress bar
        logger=logger,
    )

    # -----------------------------
    # Run arena
    # -----------------------------

    arena = ba.BitBullyArena()
    result = arena.run(cfg)
    return result

In [18]:
from techdays26 import utils

commit_sha = utils.get_commit_hash(".")
reqs = utils.get_requirements_string()

In [19]:
from pathlib import Path
from techdays26.torch_board import BoardBatch
import time

device = "cpu"
pre_trained_model: str | None = None
train_folder: str | Path = "./train_run"

n_evaluate = 1000
n_steps = 10000
lr_initial = 1e-4
lr_final = 1e-5
gamma = 0.9999
B = 1000
epsilon = 0.1

train_folder = Path(train_folder)
train_folder.mkdir(parents=True, exist_ok=True)

# load pre-trained, if desired:
if pre_trained_model:
    ntuple_net = NTupleNetwork.load(
        pre_trained_model,
        device=device,
    )
    # ntuple_net.eval() # DO NOT DO DURING TRAINING
else:
    ntuple_net = NTupleNetwork(n_tuple_list=NTUPLE_BITIDX_LIST).to(device)

torch_board = BoardBatch.empty(B, device)

assert ntuple_net.training, "Model should be in training mode!"

opt = torch.optim.Adam(ntuple_net.parameters(), lr=lr_initial)  # used to be 1e-3
scheduler = torch.optim.lr_scheduler.LambdaLR(
    opt,
    lr_lambda=lambda step: (
        (lr_final / lr_initial) + (1 - lr_final / lr_initial) * pow(gamma, step)
    ),
)
# scheduler = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=0.9999)

all_results = []
losses = []
last_eval_time = time.time()
for step in range(n_steps):
    V_old = ntuple_net(torch_board)
    randomize = (
        torch.rand((B,), device=torch_board.all_tokens.device) < epsilon
    )  # [B] bool

    with torch.no_grad():
        best_mv, V_new = best_afterstate_values(
            torch_board,
            ntuple_net,
            moves_mask=None,
            randomize=randomize,
            use_non_losing=False,
            no_grad=True,
        )

    # At any time, any move has to be legal:
    legal = torch_board.play_masks(best_mv)
    done = torch_board.done()

    # Update only for afterstates which are terminal states or not random moves
    # TODO: Maybe we can always update. Should not make a difference for epsilon=0.1
    update_mask = done | (~randomize)

    loss = torch.nn.functional.mse_loss(V_old[update_mask], V_new[update_mask])

    opt.zero_grad(set_to_none=True)
    loss.backward()

    # Snapshot W before optimizer step to compute ||ΔW|| / ||W||
    if step % 100 == 0:
        with torch.no_grad():
            W_mask = (ntuple_net.W != 0.0)
            W_before = ntuple_net.W[W_mask]  # only consider non-zero weights for norm
            W_before = W_before.data.clone()
            W_norm = W_before.norm().item()

    opt.step()
    scheduler.step()


    losses.append(loss.item())
    if step % (n_evaluate / 10) == 0:
        with torch.no_grad():
            # Compute relative weight update
            W_after = ntuple_net.W[W_mask]
            delta_W_norm = (W_after.data - W_before).norm().item()
            rel_update = delta_W_norm / W_norm if W_norm > 0 else float("inf")
        lr = opt.param_groups[0]["lr"]
        print(
            f"step {step:6d} | lr = {lr:.3e} | loss = {loss.item():.6f} | ||ΔW||/||W|| = {rel_update:.6e} | V_old (min/max): {V_old.min().item():.3f}/{V_old.max().item():.3f} | V_old (mean±std): {V_old.mean().item():.3f}±{V_old.std().item():.3f} | Avg. games done: {done.float().mean(): .2f}"
        )
        mv_left = torch_board.moves_left
        print(
            f"Moves left: min/max: {mv_left.min().item()}/{mv_left.max().item()}. mean ± std: {mv_left.float().mean().item(): .2f}±{mv_left.float().std().item():.2f}.\n"
        )

    if step % n_evaluate == 0:
        print("evaluate...")
        # Save the weights to a file
        weights_path = (
            f"{train_folder}/step_{step}_model_weights_loss_{loss.item():.3f}.pt"
        )
        ntuple_net.save(weights_path)
        td_agent = (
            TDConnect4AgentTorch(  # TODO: Allow to pass object (or copy) directly
                model_path=weights_path
            )
        )
        result = evaluate(td_agent, ba.RandomAgent(), bitbully_agents)
        result.save_json(f"{train_folder}/step_{step}_arena_result.json")

        out_file = Path(train_folder / "0_log.txt")

        with out_file.open("a", encoding="utf-8") as f:
            if step == 0:
                f.write(f"Base model: {pre_trained_model}\n")
                f.write(f"Git commit SHA: {commit_sha}\n")
                f.write(f"Requirements: {reqs}\n")
                f.write("\n--- Training Settings ---\n")
                f.write(f"device: {device}\n")
                f.write(f"batch_size (B): {B}\n")
                f.write(f"n_steps: {n_steps}\n")
                f.write(f"n_evaluate: {n_evaluate}\n")
                f.write(f"lr_initial: {lr_initial}\n")
                f.write(f"lr_final: {lr_final}\n")
                f.write(f"gamma (lr decay): {gamma}\n")
                f.write(f"epsilon (exploration): {epsilon}\n")
                f.write(
                    f"optimizer: {type(opt).__name__} (betas={opt.defaults['betas']}, eps={opt.defaults['eps']}, weight_decay={opt.defaults['weight_decay']})\n"
                )
                f.write(
                    f"scheduler: LambdaLR (lr_final/lr_initial + (1 - lr_final/lr_initial) * gamma^step)\n"
                )
                f.write(f"loss: MSE\n")
                f.write(f"use_non_losing: False\n")
                f.write(f"activation: tanh\n")
                f.write(f"n_tuples (M): {ntuple_net.M}\n")
                f.write(f"tuple_length (N): {ntuple_net.N}\n")
                f.write(f"LUT_size (K=4^N): {ntuple_net.K}\n")
                f.write(
                    f"total_params: {sum(p.numel() for p in ntuple_net.parameters())}\n"
                )
                f.write(f"mirror_symmetry: True\n")
                f.write("--- End Training Settings ---\n\n")
                f.write(get_table_legend() + "\n\n")
            lr = opt.param_groups[0]["lr"]
            grad = ntuple_net.W.grad
            grad = grad[grad != 0.0]
            upd = update_mask.float().mean().item()
            term = done.float().mean().item()
            n_wins = torch_board.has_win().sum().item()
            rand = randomize.float().mean().item()
            mv_left = torch_board.moves_left
            f.write("=====================================================" * 2 + "\n")
            f.write(
                f"step {step:6d} | time: {time.time() - last_eval_time: .1f} s | lr = {lr:.3e} | loss = {loss.item():.5f}\n\n"
            )
            f.write(f"Avg. games done: {done.float().mean(): .2f}")
            f.write(
                f"V_old (min/max): {V_old.min().item():.3f}/{V_old.max().item():.3f} | V_old (mean±std): {V_old.mean().item():.3f}±{V_old.std().item():.3f} |V_old| (mean±std): {V_old.abs().mean().item():.3f}±{V_old.abs().std().item():.3f}\n"
            )
            f.write(
                f"Gradient ∇W: shape: {grad.shape[0]}. min/max: {grad.min().item():.3f}/{grad.max().item():.3f}. mean ± std: {grad.mean().item():.3f}±{grad.std().item():.3f}.\n"
            )
            f.write(
                f"||ΔW||/||W|| = {rel_update:.6e} | ||ΔW|| = {delta_W_norm:.6e} | ||W|| = {W_norm:.6e}\n"
            )
            f.write(
                f"update_frac {upd: .2f}. done_frac: {term: .2f}. randomize_frac: {rand:.2f}. n_wins: {n_wins}\n"
            )
            f.write(
                f"Moves left: min/max: {mv_left.min().item()}/{mv_left.max().item()}. mean ± std: {mv_left.float().mean().item(): .2f}±{mv_left.float().std().item():.2f}.\n"
            )
            f.write(format_aggregate_table(result))
            f.write(
                "=====================================================" * 2 + "\n\n"
            )

        last_eval_time = time.time()

    if done.any():
        torch_board.reset(done)

step      0 | lr = 9.999e-05 | loss = 0.000000 | ||ΔW||/||W|| = inf | V_old (min/max): 0.000/0.000 | V_old (mean±std): 0.000±0.000 | Avg. games done:  0.00
Moves left: min/max: 41/41. mean ± std:  41.00±0.00.

evaluate...
step    100 | lr = 9.910e-05 | loss = 0.042225 | ||ΔW||/||W|| = 2.417283e-02 | V_old (min/max): -0.339/0.307 | V_old (mean±std): 0.016±0.114 | Avg. games done:  0.05
Moves left: min/max: 2/41. mean ± std:  30.30±7.50.

step    200 | lr = 9.821e-05 | loss = 0.049550 | ||ΔW||/||W|| = 1.482806e-02 | V_old (min/max): -0.322/0.428 | V_old (mean±std): 0.018±0.118 | Avg. games done:  0.05
Moves left: min/max: 3/41. mean ± std:  29.74±8.34.

step    300 | lr = 9.733e-05 | loss = 0.029759 | ||ΔW||/||W|| = 1.177726e-02 | V_old (min/max): -0.227/0.575 | V_old (mean±std): 0.229±0.115 | Avg. games done:  0.04
Moves left: min/max: 1/41. mean ± std:  30.12±8.95.

step    400 | lr = 9.646e-05 | loss = 0.035442 | ||ΔW||/||W|| = 1.026957e-02 | V_old (min/max): -0.275/0.637 | V_old (mea

KeyboardInterrupt: 

In [27]:
td_agent = TDConnect4AgentTorch(
            model_path="/home/mthill/techdays26/train_run/step_6000_model_weights_loss_0.016.pt"
        )

agents: dict[str, Connect4Agent] = {
    "BitBully-max-depth": bitbully_agent_max_depth,
    "TD-Agent": td_agent,
    "BitBully-4-ply": bitbully_agents["bitbully-8ply"],
}

In [28]:
from bitbully.gui_c4 import GuiC4

%matplotlib ipympl

c4gui = GuiC4(agents=agents, autoplay=True)

# Display everything
display(c4gui.get_widget())

AppLayout(children=(HBox(children=(VBox(children=(Button(description='🔄', layout=Layout(height='55.33333333333…

In [ ]:
result = evaluate(td_agent, RandomAgent(), bitbully_agent_max_depth)

In [ ]:
print(format_aggregate_table(result))

In [ ]:
# =============================================================================
# Tests for NTupleNetwork & best_afterstate_values
# =============================================================================
# Requires: cell-6 (NTUPLE_BITIDX_LIST), cell-7 (NTupleNetwork, best_afterstate_values),
#           cell-9 (bbc import)
# Run these BEFORE training (or restart kernel + run cells 1-9 + this cell).

import os
import tempfile

import bitbully.bitbully_core as bbc
import numpy as np
import torch

from techdays26.ntuples import NTUPLE_BITIDX_LIST
from techdays26.torch_board import BoardBatch, move_mask_to_column

DEVICE = "cpu"


def _mk_net(*, random_weights: bool = False) -> NTupleNetwork:
    """Create a fresh NTupleNetwork, optionally with small random weights."""
    net = NTupleNetwork(n_tuple_list=NTUPLE_BITIDX_LIST)
    if random_weights:
        with torch.no_grad():
            net.W.normal_(0, 0.05)
    net.eval()
    return net


def _build_midgame_boards(B: int, *, n_rounds: int = 15, seed: int = 42) -> BoardBatch:
    """Build a batch of diverse mid-game positions by playing random moves."""
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    board = BoardBatch.empty(B, DEVICE)
    for _ in range(n_rounds):
        actions = torch.randint(0, 7, (B,), device=DEVICE, generator=g)
        board.play_columns(actions)
        board.reset(board.done())
    return board


def _build_synced_boards(
    B: int, *, n_rounds: int = 15, seed: int = 42
) -> tuple[BoardBatch, list[bbc.BoardCore]]:
    """Build mid-game boards with a synced list of BoardCore references."""
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    board = BoardBatch.empty(B, DEVICE)
    cores = [bbc.BoardCore() for _ in range(B)]
    for _ in range(n_rounds):
        actions = torch.randint(0, 7, (B,), device=DEVICE, generator=g)
        board.play_columns(actions)
        for i in range(B):
            cores[i].play(int(actions[i].item()))
        done = board.done()
        board.reset(done)
        for i in range(B):
            if cores[i].hasWin() or cores[i].movesLeft() <= 0:
                cores[i].setRawState(0, 0, 42)
    return board, cores


# ---------------------------------------------------------------------------
# 1) NTupleNetwork: output range
# ---------------------------------------------------------------------------
def test_ntuple_network_output_in_tanh_range():
    """forward() must return values in [-1, 1] (tanh activation)."""
    net = _mk_net(random_weights=True)
    board = _build_midgame_boards(256)

    with torch.no_grad():
        vals = net(board)

    assert vals.shape == (256,)
    assert vals.min() >= -1.0 and vals.max() <= 1.0, (
        f"Values outside [-1, 1]: min={vals.min():.6f}, max={vals.max():.6f}"
    )
    print("PASSED: test_ntuple_network_output_in_tanh_range")


# ---------------------------------------------------------------------------
# 2) NTupleNetwork: zero weights -> zero output
# ---------------------------------------------------------------------------
def test_ntuple_network_zero_weights_give_zero():
    """With W=0 the network must output exactly 0 for every board."""
    net = _mk_net(random_weights=False)  # W is all zeros
    board = _build_midgame_boards(128)

    with torch.no_grad():
        vals = net(board)

    assert (vals == 0.0).all(), (
        f"Expected all zeros, got min={vals.min()}, max={vals.max()}"
    )
    print("PASSED: test_ntuple_network_zero_weights_give_zero")


# ---------------------------------------------------------------------------
# 3) NTupleNetwork: mirror symmetry (value of board == value of mirror)
# ---------------------------------------------------------------------------
def test_ntuple_network_mirror_symmetry():
    """The network uses W + W_mirror, so v(board) == v(mirror(board))."""
    net = _mk_net(random_weights=True)
    board = _build_midgame_boards(256)
    mir = board.mirror()

    with torch.no_grad():
        v_orig = net(board)
        v_mir = net(mir)

    assert torch.allclose(v_orig, v_mir, atol=1e-6), (
        f"Mirror asymmetry: max diff = {(v_orig - v_mir).abs().max():.8f}"
    )
    print("PASSED: test_ntuple_network_mirror_symmetry")


# ---------------------------------------------------------------------------
# 4) NTupleNetwork: save / load roundtrip
# ---------------------------------------------------------------------------
def test_ntuple_network_save_load_roundtrip():
    """Save + load must produce bit-identical forward pass results."""
    net = _mk_net(random_weights=True)
    board = _build_midgame_boards(64)

    with torch.no_grad():
        v_before = net(board).clone()

    with tempfile.NamedTemporaryFile(suffix=".pt", delete=False) as f:
        path = f.name
    try:
        net.save(path)
        net2 = NTupleNetwork.load(path, device="cpu")
        net2.eval()
        with torch.no_grad():
            v_after = net2(board)
        assert torch.equal(v_before, v_after), "Forward pass differs after save/load"
    finally:
        os.remove(path)
    print("PASSED: test_ntuple_network_save_load_roundtrip")


# ---------------------------------------------------------------------------
# 5) best_afterstate_values: returned moves are legal one-hot masks
# ---------------------------------------------------------------------------
def test_bav_returns_legal_onehot_moves():
    """Every non-zero move from best_afterstate_values must be a legal one-hot landing square."""
    net = _mk_net(random_weights=True)
    board = _build_midgame_boards(512)

    with torch.no_grad():
        best_mv, best_val = best_afterstate_values(board, net, use_non_losing=False)

    legal = board.generate_legal_moves()
    active = best_mv != 0

    # Subset of legal moves
    assert ((best_mv[active] & legal[active]) == best_mv[active]).all(), (
        "Some returned moves are not legal landing squares"
    )
    # One-hot (single bit set) or zero
    one_hot_or_zero = (best_mv == 0) | ((best_mv & (best_mv - 1)) == 0)
    assert one_hot_or_zero.all(), "Some returned moves have multiple bits set"
    print("PASSED: test_bav_returns_legal_onehot_moves")


# ---------------------------------------------------------------------------
# 6) best_afterstate_values: moves are within non-losing set
# ---------------------------------------------------------------------------
def test_bav_moves_subset_of_non_losing():
    """With use_non_losing=True, returned moves must be within non-losing moves."""
    net = _mk_net(random_weights=True)
    board = _build_midgame_boards(512)

    nl_moves = board.generate_non_losing_moves()

    with torch.no_grad():
        best_mv, _ = best_afterstate_values(board, net, use_non_losing=True)

    active = best_mv != 0
    assert ((best_mv[active] & nl_moves[active]) == best_mv[active]).all(), (
        "Some returned moves are outside the non-losing move set"
    )
    print("PASSED: test_bav_moves_subset_of_non_losing")


# ---------------------------------------------------------------------------
# 7) best_afterstate_values: randomize=True still returns legal moves
# ---------------------------------------------------------------------------
def test_bav_random_mode_still_legal():
    """With randomize=all True, returned moves must still be legal and one-hot."""
    net = _mk_net(random_weights=True)
    board = _build_midgame_boards(512)

    randomize = torch.ones(512, dtype=torch.bool)
    with torch.no_grad():
        rand_mv, rand_val = best_afterstate_values(
            board,
            net,
            randomize=randomize,
            use_non_losing=False,
        )

    legal = board.generate_legal_moves()
    active = rand_mv != 0
    assert ((rand_mv[active] & legal[active]) == rand_mv[active]).all(), (
        "Random moves are not all legal"
    )
    one_hot_or_zero = (rand_mv == 0) | ((rand_mv & (rand_mv - 1)) == 0)
    assert one_hot_or_zero.all(), "Random moves are not one-hot"
    print("PASSED: test_bav_random_mode_still_legal")


# ---------------------------------------------------------------------------
# 8) best_afterstate_values: terminal afterstates have correct reward as value
# ---------------------------------------------------------------------------
def test_bav_terminal_rewards():
    """For terminal afterstates, V_new must be the exact game reward (+1/-1/0)."""
    net = _mk_net(random_weights=True)
    board = _build_midgame_boards(2048, n_rounds=20)

    with torch.no_grad():
        best_mv, best_val = best_afterstate_values(board, net, use_non_losing=False)

    # Apply the chosen moves to copies
    tmp = BoardBatch(
        all_tokens=board.all_tokens.clone(),
        active_tokens=board.active_tokens.clone(),
        moves_left=board.moves_left.clone(),
    )
    tmp.play_masks(best_mv)
    done = tmp.done()
    reward = tmp.reward()

    n_terminal = done.sum().item()
    assert n_terminal > 0, "No terminal afterstates found; increase n_rounds or B"

    # For terminal states, best_val must match the reward
    terminal_vals = best_val[done]
    terminal_rewards = reward[done]
    assert torch.equal(terminal_vals, terminal_rewards), (
        f"Terminal values don't match rewards.\n"
        f"  vals    = {terminal_vals[:10].tolist()}\n"
        f"  rewards = {terminal_rewards[:10].tolist()}"
    )

    # Verify specific reward values
    has_win = tmp.has_win()
    yellow_won = done & has_win & ((tmp.moves_left.to(torch.int64) & 1) == 1)
    red_won = done & has_win & ((tmp.moves_left.to(torch.int64) & 1) == 0)
    draw = done & (~has_win) & (tmp.moves_left == 0)

    if yellow_won.any():
        assert (best_val[yellow_won] == 1.0).all(), "Yellow wins should have value +1.0"
    if red_won.any():
        assert (best_val[red_won] == -1.0).all(), "Red wins should have value -1.0"
    if draw.any():
        assert (best_val[draw] == 0.0).all(), "Draws should have value 0.0"

    print(
        f"  Verified {n_terminal} terminal states: "
        f"{yellow_won.sum()} yellow wins, {red_won.sum()} red wins, {draw.sum()} draws"
    )
    print("PASSED: test_bav_terminal_rewards")


# ---------------------------------------------------------------------------
# 9) best_afterstate_values: cross-check with BoardCore (the original disabled code)
# ---------------------------------------------------------------------------
def test_bav_consistency_with_core():
    """Play best_afterstate_values moves on BoardCore in parallel and verify state match.

    This is the proper test version of the `if False:` verification block.
    """
    net = _mk_net(random_weights=True)
    B = 200
    board, cores = _build_synced_boards(B, n_rounds=15)

    with torch.no_grad():
        best_mv, V_new = best_afterstate_values(
            board,
            net,
            use_non_losing=False,
            no_grad=True,
        )

    for b_idx in range(B):
        mv = int(best_mv[b_idx].item())
        if mv == 0:
            continue  # board already terminal or forced-loss (no moves)

        col = move_mask_to_column(mv)
        assert cores[b_idx].isLegalMove(col), (
            f"Board {b_idx}: move column {col} is not legal in core"
        )

        # Play on core
        ok = cores[b_idx].play(col)
        assert ok, f"Board {b_idx}: core.play({col}) returned False"
        bb_done = cores[b_idx].hasWin() or cores[b_idx].movesLeft() <= 0

        # Play on BoardBatch copy to check state equivalence
        tmp = BoardBatch(
            all_tokens=board.all_tokens[b_idx : b_idx + 1].clone(),
            active_tokens=board.active_tokens[b_idx : b_idx + 1].clone(),
            moves_left=board.moves_left[b_idx : b_idx + 1].clone(),
        )
        tmp.play_masks(torch.tensor([mv], dtype=torch.int64))

        # State equivalence
        a_core, b_core, m_core = cores[b_idx].rawState()
        assert int(a_core) == int(tmp.all_tokens[0].item()), (
            f"Board {b_idx}: all_tokens mismatch"
        )
        assert int(b_core) == int(tmp.active_tokens[0].item()), (
            f"Board {b_idx}: active_tokens mismatch"
        )
        assert int(m_core) == int(tmp.moves_left[0].item()), (
            f"Board {b_idx}: moves_left mismatch"
        )

        tmp_done = bool(tmp.done()[0].item())
        assert bb_done == tmp_done, (
            f"Board {b_idx}: done mismatch (core={bb_done}, batch={tmp_done})"
        )

        # Terminal reward check (same logic as the original disabled code)
        if bb_done:
            v = float(V_new[b_idx].item())
            if cores[b_idx].hasWin() and cores[b_idx].movesLeft() % 2 == 0:
                assert v == -1.0, f"Board {b_idx}: red win should be -1, got {v}"
            elif cores[b_idx].hasWin() and cores[b_idx].movesLeft() % 2 == 1:
                assert v == 1.0, f"Board {b_idx}: yellow win should be +1, got {v}"
            elif cores[b_idx].movesLeft() <= 0:
                assert v == 0.0, f"Board {b_idx}: draw should be 0, got {v}"

        # Reset core if done (to match the original verification pattern)
        if bb_done:
            cores[b_idx].setRawState(0, 0, 42)

    print("PASSED: test_bav_consistency_with_core")


# ---------------------------------------------------------------------------
# 10) best_afterstate_values: greedy choice is optimal (accounting for tiebreak noise)
# ---------------------------------------------------------------------------
def test_bav_greedy_is_optimal():
    """The greedy move's un-noised value must be within noise tolerance of the true optimum.

    best_afterstate_values adds tiebreak noise eps=1e-4 * randn, so we allow ~5e-4 tolerance.
    """
    net = _mk_net(random_weights=True)
    B = 128
    board = _build_midgame_boards(B, n_rounds=12)

    with torch.no_grad():
        best_mv, best_val = best_afterstate_values(board, net, use_non_losing=False)

    yellow_to_move = (board.moves_left.to(torch.int64) & 1) == 0

    # Compute the true best value (without noise) by enumerating all legal moves
    legal_mask = board.generate_legal_moves()
    neg_inf = torch.full((B,), float("-inf"))
    pos_inf = torch.full((B,), float("inf"))
    true_best = torch.where(yellow_to_move, neg_inf, pos_inf).to(torch.float32)

    for mv in board.iter_move_masks(legal_mask):
        active = mv != 0
        if not active.any():
            break
        tmp = BoardBatch(
            all_tokens=board.all_tokens.clone(),
            active_tokens=board.active_tokens.clone(),
            moves_left=board.moves_left.clone(),
        )
        tmp.play_masks(mv)
        r = tmp.reward().to(torch.float32)
        with torch.no_grad():
            v = net(tmp).to(torch.float32)
        val = torch.where(torch.isnan(r), v, r)

        better = torch.where(yellow_to_move, val > true_best, val < true_best) & active
        true_best = torch.where(better, val, true_best)

    # The chosen value (noise-free recomputed) should be close to true_best
    # Recompute the chosen move's noise-free value
    tmp_chosen = BoardBatch(
        all_tokens=board.all_tokens.clone(),
        active_tokens=board.active_tokens.clone(),
        moves_left=board.moves_left.clone(),
    )
    tmp_chosen.play_masks(best_mv)
    r_chosen = tmp_chosen.reward().to(torch.float32)
    with torch.no_grad():
        v_chosen = net(tmp_chosen).to(torch.float32)
    chosen_noisefree = torch.where(torch.isnan(r_chosen), v_chosen, r_chosen)

    has_move = best_mv != 0
    tol = 5e-4  # noise is ~1e-4 * randn; 5-sigma margin

    yellow_ok = ~has_move | ~yellow_to_move | (chosen_noisefree >= true_best - tol)
    red_ok = ~has_move | yellow_to_move | (chosen_noisefree <= true_best + tol)

    assert yellow_ok.all(), (
        f"Yellow suboptimality beyond tolerance: "
        f"worst gap = {(true_best[yellow_to_move & has_move] - chosen_noisefree[yellow_to_move & has_move]).max():.6f}"
    )
    assert red_ok.all(), (
        f"Red suboptimality beyond tolerance: "
        f"worst gap = {(chosen_noisefree[~yellow_to_move & has_move] - true_best[~yellow_to_move & has_move]).max():.6f}"
    )

    print(f"  Checked {has_move.sum()} boards with moves")
    print("PASSED: test_bav_greedy_is_optimal")


# ---------------------------------------------------------------------------
# 11) Training-loop invariants: moves are always legal, done boards have rewards
# ---------------------------------------------------------------------------
def test_training_loop_invariants():
    """Simulate a few training steps and verify the two `if False:` guards from the loop:
    1) All returned moves are legal (play_masks returns True)
    2) Done boards always have a non-NaN reward
    """
    net = _mk_net(random_weights=True)
    B = 1024
    board = BoardBatch.empty(B, DEVICE)
    epsilon = 0.1

    for step in range(50):
        randomize = torch.rand((B,), device=DEVICE) < epsilon

        with torch.no_grad():
            best_mv, V_new = best_afterstate_values(
                board,
                net,
                randomize=randomize,
                use_non_losing=False,
                no_grad=True,
            )

        legal = board.play_masks(best_mv)

        # Invariant 1: all moves should be accepted as legal
        assert legal.all(), f"Step {step}: {(~legal).sum()} illegal moves returned"

        # Invariant 2: done boards must have non-NaN reward
        done = board.done()
        reward = board.reward()
        done_nan = done & torch.isnan(reward)
        assert not done_nan.any(), (
            f"Step {step}: {done_nan.sum()} done boards have NaN reward"
        )

        board.reset(done)

    print("PASSED: test_training_loop_invariants")


# ---------------------------------------------------------------------------
# 12) NTupleNetwork: gradient flows through all components
# ---------------------------------------------------------------------------
def test_ntuple_network_gradient_flow():
    """A single forward + backward step must produce non-zero gradients on W."""
    net = NTupleNetwork(n_tuple_list=NTUPLE_BITIDX_LIST)
    # Small random weights to avoid zero-gradient at tanh saturation
    with torch.no_grad():
        net.W.normal_(0, 0.01)
    net.train()

    board = _build_midgame_boards(64)
    vals = net(board)
    loss = vals.sum()
    loss.backward()

    assert net.W.grad is not None, "W.grad is None"
    assert (net.W.grad != 0).any(), "W.grad is all zeros (no gradient flow)"
    print(f"  Non-zero grad entries: {(net.W.grad != 0).sum()} / {net.W.grad.numel()}")
    print("PASSED: test_ntuple_network_gradient_flow")


# ===========================================================================
# Run all tests
# ===========================================================================
def run_all_tests():
    tests = [
        test_ntuple_network_output_in_tanh_range,
        test_ntuple_network_zero_weights_give_zero,
        test_ntuple_network_mirror_symmetry,
        test_ntuple_network_save_load_roundtrip,
        test_ntuple_network_gradient_flow,
        test_bav_returns_legal_onehot_moves,
        test_bav_moves_subset_of_non_losing,
        test_bav_random_mode_still_legal,
        test_bav_terminal_rewards,
        test_bav_consistency_with_core,
        test_bav_greedy_is_optimal,
        test_training_loop_invariants,
    ]

    passed, failed = 0, 0
    for t in tests:
        try:
            t()
            passed += 1
        except Exception as e:
            print(f"FAILED: {t.__name__}: {e}")
            import traceback

            traceback.print_exc()
            failed += 1

    print(f"\n{'=' * 60}")
    print(f"Results: {passed} passed, {failed} failed, {passed + failed} total")
    if failed == 0:
        print("All tests passed!")


run_all_tests()